In [30]:
import os
import sys

# 1. Force the notebook to run from your project workspace root
project_root = r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression"
os.chdir(project_root)

# 2. Add the 'src' folder to Python's system path so it can see 'Linear_regression_01'
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print("Current Working Directory:", os.getcwd())
print("System path updated. Available modules:", os.listdir(src_path))

Current Working Directory: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression
System path updated. Available modules: ['Linear_regression_01', 'Linear_regression_01.egg-info']


In [31]:
import os
print(os.getcwd())

C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression


In [32]:
import sys
print(sys.path)

['C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\python310.zip', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\DLLs', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire', '', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages', 'C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\linear_regression\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\win32', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\win32\\lib', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\pythonwin']


In [33]:
import os

print(os.path.exists("../src"))
print(os.path.exists("../src/Linear_regression_01"))

False
False


In [34]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(PROJECT_ROOT)
print(SRC_PATH)
print(sys.path[:3])

C:\Users\Greesha Vaishnavi\Desktop\dsprojects
C:\Users\Greesha Vaishnavi\Desktop\dsprojects\src
['C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\python310.zip', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\DLLs']


In [35]:
import box
print(box.__version__)

7.4.1


In [36]:
# entity 

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    STATUS_FILE:Path
    transformed_train_path:Path
    transformed_test_path:Path
    preprocessor_path:Path
    trained_model_file_path:Path
    metric_file_path:Path
    threshold:float

In [37]:
from Linear_regression_01.entity.config_entity import ModelEvaluationConfig
from Linear_regression_01.utils.common import read_yaml, create_directories
from Linear_regression_01.constant import CONFIG_FILE_PATH
from Linear_regression_01.constant import PARAMS_FILE_PATH
from Linear_regression_01.constant import SCHEMA_FILE_PATH
from Linear_regression_01.config.configuration import ConfigurationManager


In [38]:
# Configuration Manager

class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        config = self.config.model_trainer

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:

        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            transformed_train_path=config.transformed_train_path,
            transformed_test_path=config.transformed_test_path,
            preprocessor_path=config.preprocessor_path,
            trained_model_file_path=config.trained_model_file_path,
            metric_file_path=config.metric_file_path,
            threshold=self.params.model_evaluation.threshold
        )

        return model_evaluation_config

In [39]:
import os
import joblib
import json
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)

from Linear_regression_01.entity.config_entity import ModelEvaluationConfig
from Linear_regression_01.logging import logger

In [40]:
# components

class ModelEvaluation:

    def __init__(self,config:ModelEvaluationConfig):
        self.config=config

    def eval_metrics(self, actual, pred):
        mae = mean_absolute_error(actual, pred)
        mse = mean_squared_error(actual, pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(actual, pred)

        return mae, mse, rmse, r2

    def save_metrics(self, metrics):
        os.makedirs(self.config.root_dir, exist_ok=True)

        with open(self.config.metric_file_path, "w") as f:
            json.dump(metrics, f, indent=4)

    def initiate_model_evaluation(self):

        # Load trained model
        model = joblib.load(self.config.trained_model_file_path)

        # Load transformed test data
        test_arr = np.load(self.config.transformed_test_path)

        # Split X and y
        X_test = test_arr[:, :-1]
        y_test = test_arr[:, -1]

        # Prediction
        prediction = model.predict(X_test)

        # Calculate metrics
        mae, mse, rmse, r2 = self.eval_metrics(y_test, prediction)

        metrics = {
            "MAE": float(mae),
            "MSE": float(mse),
            "RMSE": float(rmse),
            "R2 Score": float(r2)
        }

        # Save metrics
        self.save_metrics(metrics)

        # Check threshold
        if r2 >= self.config.threshold:
            print("Model Accepted")
            return True

        else:
            print("Model Rejected")
            return False    

                                   

In [41]:
#pipeline

STAGE_NAME = "Model Evaluation Stage"

class ModelEvaluationTrainingPipeline:
    def __int__(self):
        pass

    def main(Self):
        config = ConfigurationManager()

        model_evaluation_config = (
            config.get_model_evaluation_config()
        )

        model_evaluation = ModelEvaluation(
            config = model_evaluation_config
        )

        model_evaluation.initiate_model_evaluation()

In [42]:
obj = ModelEvaluationTrainingPipeline()

obj.main()

[2026-07-27 16:06:55,774: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-27 16:06:55,776: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-27 16:06:55,780: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-07-27 16:06:55,780: INFO: common: created directory at artifacts]
[2026-07-27 16:06:55,786: INFO: common: created directory at artifacts/model_evaluation]
Model Accepted
